# 05 — The Wasserstein Distance

Promote the Kantorovich *cost* into a genuine *distance* — one that, unlike the KL divergence, listens to the geometry of the ground space.

**Prerequisites:** `04_birkhoff_and_assignment`.

**What you'll be able to do**
- State the Wasserstein-$p$ distance $W_p(\mu, \nu)$ and verify its metric axioms numerically.
- Contrast $W_p$ with KL — watch $W_2$ grow linearly with displacement while KL diverges.
- Explain why a geometry-aware distance is the right tool when the ground space carries meaning.

In the last arc you turned transport into a solvable linear program. Its optimal *cost* now becomes something stronger: a genuine **distance** between distributions. The Wasserstein distance is a true metric, and — unlike the KL divergence of Module 2 — it hears the geometry of the ground space. That single property is why optimal transport matters for signals, images, and the quantum states ahead.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from qot_course import viz
from qot_course.infotheory.classical import kl_divergence
from qot_course.transport.wasserstein import wasserstein_1d

np.random.seed(42)
viz.use_course_style()

## 1. Definition and the metric axioms

Take two probability measures $\mu, \nu$ on $\mathbb{R}^d$ with the Euclidean ground cost $c(x, y) = \|x - y\|$. The **Wasserstein-$p$ distance** ($p \ge 1$) is the optimal transport cost, raised to the power $1/p$:

$$ W_p(\mu, \nu) = \Bigl( \inf_{\pi \in \Pi(\mu, \nu)} \int \|x - y\|^p\, \mathrm{d}\pi(x, y) \Bigr)^{1/p}, $$

where $\Pi(\mu, \nu)$ is the set of couplings — the transportation polytope of the last arc. The infimum is attained (the polytope is compact), so $W_p$ is a number we can compute for any pair.

The raising to $1/p$ is what makes it a genuine **metric** on distributions of finite $p$-th moment (Villani, 2003, Thm. 7.3): (i) $W_p \ge 0$, with $W_p(\mu, \nu) = 0$ iff $\mu = \nu$; (ii) symmetry $W_p(\mu, \nu) = W_p(\nu, \mu)$; (iii) the **triangle inequality** $W_p(\mu, \rho) \le W_p(\mu, \nu) + W_p(\nu, \rho)$. Let's check all three on the line.

In [ ]:
mu  = (np.array([0.0, 1.0, 2.0]), np.array([0.2, 0.3, 0.5]))
nu  = (np.array([0.5, 1.5, 2.5]), np.array([0.4, 0.4, 0.2]))
rho = (np.array([-0.5, 0.0, 3.0]), np.array([0.1, 0.5, 0.4]))

d_mu_mu  = wasserstein_1d(*mu,  *mu,  p=2)
d_mu_nu  = wasserstein_1d(*mu,  *nu,  p=2)
d_nu_mu  = wasserstein_1d(*nu,  *mu,  p=2)
d_mu_rho = wasserstein_1d(*mu,  *rho, p=2)
d_nu_rho = wasserstein_1d(*nu,  *rho, p=2)

print(f"identity:  W2(mu, mu)  = {d_mu_mu:.6f}                          (-> 0)")
print(f"symmetry:  W2(mu, nu)  = {d_mu_nu:.4f}    W2(nu, mu) = {d_nu_mu:.4f}")
print(f"triangle:  W2(mu, rho) = {d_mu_rho:.4f}   <=   W2(mu,nu) + W2(nu,rho) = {d_mu_nu + d_nu_rho:.4f}")

**Read the output.** All three axioms hold: $W_2(\mu, \mu) = 0$ to machine precision, the two directions agree exactly (symmetry), and the route through $\nu$ is no shorter than the direct path (triangle). These are precisely the properties KL lacks — KL is asymmetric and violates the triangle inequality — which is why $W_p$, not KL, deserves the name *distance*.

## 2. Why $W_p$ respects geometry

Here is the cleanest contrast with KL. Take a single bump $\mu$ and slide it a distance $d$ along the line. The Wasserstein distance grows in step with the move,

$$ W_p(\mu,\ \mu_{\text{shifted by } d}) = d, $$

because it literally measures how far the mass travelled. KL, by contrast, is blind to the ground space: once the shifted bump stops overlapping the original, KL is dominated by the mismatch in the tails and races off to infinity. Let's watch both at once.

In [ ]:
grid = np.linspace(-1.0, 11.0, 600)
dx = grid[1] - grid[0]

def density(center, sigma):
    """Gaussian bump on the grid, normalised to a probability density."""
    pdf = np.exp(-0.5 * ((grid - center) / sigma) ** 2)
    return pdf / np.trapezoid(pdf, grid)

ref_mass = density(3.0, 0.8) * dx
ref_mass /= ref_mass.sum()

shifts = np.arange(0.0, 7.5, 0.5)
w2_curve, kl_curve = [], []
for d in shifts:
    shifted_mass = density(3.0 + d, 0.8) * dx
    shifted_mass /= shifted_mass.sum()
    w2_curve.append(wasserstein_1d(grid, ref_mass, grid, shifted_mass, p=2))
    kl_curve.append(kl_divergence(ref_mass + 1e-300, shifted_mass + 1e-300))  # bits

fig, ax = plt.subplots(figsize=(10, 4.8))
ax_kl = ax.twinx()
l1, = ax.plot(shifts, w2_curve, "-o", color=viz.SOURCE_COLOR, lw=2, label=r"$W_2(\mu, \mu_d)$  (left axis)")
l2, = ax_kl.plot(shifts, kl_curve, "-s", color=viz.TARGET_COLOR, lw=2, label=r"$D_{\mathrm{KL}}(\mu \,\|\, \mu_d)$ (right axis, log)")
ax_kl.set_yscale("log")
ax.set_xlabel("displacement  d")
ax.set_ylabel(r"$W_2$  (linear)", color=viz.SOURCE_COLOR)
ax_kl.set_ylabel(r"$D_{\mathrm{KL}}$  (log scale)", color=viz.TARGET_COLOR)
ax.set_title("W2 grows linearly in displacement; KL blows up as supports separate", pad=12)
ax.legend(handles=[l1, l2], loc="upper left")
plt.tight_layout()
plt.show()

**Read the figure.** The violet $W_2$ curve is a straight line through the origin with slope $1$ — exactly $W_2(\mu, \mu_d) = d$, the signature of a transport distance. The amber KL curve (right axis, log scale) climbs *exponentially*: once the displaced bump leaves the reference's support (around $d \gtrsim 3$) the two share almost no mass and KL diverges. Same data, two distances, two different universes. This is why optimal transport is the right metric whenever the ground space carries geometry — images, signals, neural recordings, and the density matrices we are heading toward.

## Your turn

1. **Symmetry, side by side.** Pick two distributions and compute both $W_2(\mu, \nu)$ and $W_2(\nu, \mu)$, then both $D_{\mathrm{KL}}(\mu\|\nu)$ and $D_{\mathrm{KL}}(\nu\|\mu)$. Which distance is symmetric, and by how much does KL differ?
2. **A tight triangle.** Find three distributions on the line for which the triangle inequality is *almost* an equality. What geometric arrangement makes the detour through $\nu$ nearly free?
3. **Read the slope.** In the displacement demo the $W_2$ line has slope $1$. Predict and then check the slope of $W_1(\mu, \mu_d)$ as a function of $d$. Does the power $p$ change the slope of a pure translation?

## What you built

- You stated the Wasserstein-$p$ distance and verified its three metric axioms numerically.
- You contrasted $W_2$ with KL: $W_2$ grows linearly with displacement, KL diverges once supports separate.
- You can now say *why* a geometry-aware distance is the right tool for data that lives in a meaningful space.

You have promoted a transport cost into a true geometry on distributions — a metric that measures how far mass must move. Every distance comparison in the rest of the course leans on this idea.

## What's next

Computing $W_p$ by solving the full LP is heavy. In `06_1d_quantile_closed_form` we discover that on the line it collapses to *sort and match by quantile* — an $O(n \log n)$ shortcut — and read $W_1$ off as the area between two CDFs.

## References

- C. Villani, *Topics in Optimal Transportation*, AMS (2003), ch. 2 and Thm. 7.3.
- G. Peyré & M. Cuturi, *Computational Optimal Transport*, NoW (2019), ch. 2. DOI:10.1561/2200000073
- S. S. Vallender, "Calculation of the Wasserstein distance between probability distributions on the line", *Theory Probab. Appl.* 18, 784–786 (1974).

**Previous:** `notebooks/03_ClassicalOptimalTransport/04_birkhoff_and_assignment.ipynb` · **Next:** `notebooks/03_ClassicalOptimalTransport/06_1d_quantile_closed_form.ipynb`